# Email Negotiation — GRPO Training with OpenEnv

**Environment**: Two-LLM email negotiation for enterprise deal closing.

**Seller** (trainable): Learns to negotiate via structured email actions.
**Buyer** (frozen LLM): Six stakeholder archetypes with persistent memory.

**Goal**: Close deals by addressing concerns, delivering documents, and making appropriate concessions.

---

**Setup**

In [ ]:
!pip install openenv-core[core] openai numpy trl matplotlib pandas --quiet 2>&1 | tail -5

In [ ]:
import sys, os
sys.path.insert(0, '/Users/akshaypulla/Documents/deal_room/email_negotiation')
os.chdir('/Users/akshaypulla/Documents/deal_room/email_negotiation')
print('Working directory:', os.getcwd())

---

## 1. Environment Anatomy — Deep Dive

Before training, understand every component of the environment.

In [ ]:
from server.environment import EmailNegotiationEnvironment
from models import EmailAction, EmailObservation, EmailState

env = EmailNegotiationEnvironment()
print('=== Environment ===')
print('Type:', type(env).__name__)
print('Supports concurrent sessions:', getattr(env, 'SUPPORTS_CONCURRENT_SESSIONS', False))

In [ ]:
# Inspect the three core dataclasses
print('=== EmailAction Fields ===')
for f in EmailAction.model_fields:
    print(f'  {f}: {EmailAction.model_fields[f].annotation}')

print('\n=== EmailObservation Fields ===')
for f in EmailObservation.model_fields:
    print(f'  {f}: {EmailObservation.model_fields[f].annotation}')

print('\n=== EmailState Fields ===')
for f in EmailState.model_fields:
    print(f'  {f}: {EmailState.model_fields[f].annotation}')

In [ ]:
print('\n=== Valid Action Values ===')
print('intents:   ', {'address_concern', 'offer_document', 'make_concession', 'escalate_to_exec', 'group_proposal', 'walkaway'})
print('targets:   ', {'Legal', 'Finance', 'CTO', 'TechLead', 'Procurement', 'Operations', 'ExecSponsor'})
print('tones:     ', {'formal', 'reassuring', 'urgent'})
print('docs:      ', {'DPA', 'roi_model', 'security_cert', 'implementation_timeline', 'vendor_packet'})
print('urgency:   ', {'normal', 'high'})
print('CC:        max 2 recipients (strategic influence signal)')

---

## 2. Reset & First Observation

On `reset()`, the environment:
1. Picks a random scenario (aligned / conflicted / hostile_acquisition)
2. Creates one StatefulArchetypeAgent per stakeholder
3. Returns the full inbox summary as the initial observation

In [ ]:
env = EmailNegotiationEnvironment()
obs = env.reset()

print('=== Initial Observation ===')
print(f"Deal Stage:     {obs.deal_stage}")
print(f"Progress Score: {obs.progress_score}")
print(f"Round:          {obs.round_number}/{obs.max_rounds}")
print(f"Unresolved:     {obs.unresolved_concerns}")
print(f"Reward:         {obs.reward}")
print(f"Done:           {obs.done}")

print('\n=== Inbox Summary ===')
print(obs.inbox_summary[:600])

---

## 3. Taking Steps — Action → Response → Reward

Each `step()`:
1. Validates the action (policy constraints + anti-gaming)
2. Generates a stakeholder response (frozen LLM, deterministic + stochastic)
3. Extracts reward via 3-layer pipeline (keyword → LLM features → code)
4. Computes Δ(progress_score) shaping reward
5. Updates alignment scores, concern lists, deal stage
6. Checks terminal conditions (deal_closed / veto / max_rounds)

In [ ]:
from server.email_env.progress_score import STAKEHOLDER_WEIGHTS

print('=== Stakeholder Weights (for S component) ===')
for k, v in sorted(STAKEHOLDER_WEIGHTS.items(), key=lambda x: -x[1]):
    print(f'  {k:<15}: {v:.2f}')

print('\n=== progress_score = S^0.4 × D^0.2 × G^0.2 × R^0.2 ===')
print('S = weighted stakeholder alignment')
print('D = document completeness (delivered / required)')
print('G = deal stage progress (0=initial, 1=closed)')
print('R = concern resolution (resolved / total)')

In [ ]:
# Run 10 steps with diverse actions, track every metric
env = EmailNegotiationEnvironment()
obs = env.reset()

actions_log = []
rewards_log = []
progress_log = []
stage_log = []
breakdown_log = []

action_sequence = [
    {'intent': 'offer_document', 'target': 'Legal',    'include_document': 'DPA',         'tone': 'formal'},
    {'intent': 'address_concern', 'target': 'Finance',  'include_document': 'roi_model',   'tone': 'reassuring'},
    {'intent': 'make_concession', 'target': 'Legal',    'concession_term': 'price',        'concession_value': 85000, 'tone': 'formal'},
    {'intent': 'offer_document', 'target': 'CTO',       'include_document': 'security_cert', 'tone': 'formal'},
    {'intent': 'address_concern', 'target': 'Finance',  'include_document': None,           'tone': 'formal'},
    {'intent': 'make_concession', 'target': 'CTO',      'concession_term': 'timeline',      'concession_value': 2, 'tone': 'reassuring'},
    {'intent': 'group_proposal',  'target': 'Legal',    'cc': ['Finance', 'CTO'],          'tone': 'formal'},
    {'intent': 'address_concern', 'target': 'Finance',  'include_document': None,           'tone': 'reassuring'},
    {'intent': 'offer_document', 'target': 'Legal',     'include_document': 'implementation_timeline', 'tone': 'formal'},
    {'intent': 'escalate_to_exec', 'target': 'ExecSponsor', 'tone': 'urgent'},
]

for i, base_action in enumerate(action_sequence):
    action = EmailAction(
        intent=base_action['intent'],
        target=base_action['target'],
        tone=base_action.get('tone', 'formal'),
        cc=base_action.get('cc', []),
        include_document=base_action.get('include_document'),
        concession_term=base_action.get('concession_term'),
        concession_value=base_action.get('concession_value'),
        urgency=base_action.get('urgency', 'normal'),
    )
    obs = env.step(action)
    actions_log.append(f"{action.intent}→{action.target}")
    rewards_log.append(obs.reward)
    progress_log.append(obs.progress_score)
    stage_log.append(obs.deal_stage)
    print(f'Step {i+1:2d}: {action.intent:<20} → {action.target:<12} | '
          f'r={obs.reward:+.2f} | P={obs.progress_score:.3f} | {obs.deal_stage}')

print(f'\nSum rewards: {sum(rewards_log):.3f}')
print(f'Final progress: {progress_log[-1]:.3f}')
print(f'Deals stages: {stage_log}')

---

## 4. Reward Architecture — Why It's Robust

Reward has 3 layers so the LLM NEVER directly produces a reward number:

**Layer 1 — Keyword patterns** (fast): Document names, key terms, positive/negative markers
**Layer 2 — LLM structured features** (medium): Sentiment, concerns raised/resolved, escalation
**Layer 3 — Code computes final reward** (deterministic): All shaping, clipping, gating

This means even a compromised LLM cannot inflate its own reward.

In [ ]:
from server.email_env.reward_extractor import POSITIVE_MARKERS, NEGATIVE_MARKERS, KEY_TERMS

print('=== Reward Layer 1 — Keyword Sets ===')
print('Positive markers:', POSITIVE_MARKERS)
print('Negative markers:', NEGATIVE_MARKERS)
print('Key terms:       ', KEY_TERMS)

print('\n=== Reward Magnitudes ===')
print(f'Sentiment positive:    +0.15')
print(f'Sentiment skeptical:   -0.10')
print(f'Document acknowledged: +0.10')
print(f'Engagement (questions): +0.10')
print(f'Concern resolved:      +0.20')
print(f'Concern new:           -0.10')
print(f'Objection strong:      -0.50')
print(f'CC no reply:           -0.05')
print(f'Terminal deal closed:  +3.00')
print(f'Terminal veto:         -3.00')
print(f'Terminal max rounds:    -1.50')

print('=== Shaping Reward ===')
print('Δ(progress_score) × 2.0, clipped to [-0.5, +0.5]')
print('All shaping ≤ 0.5 × terminal (agent never prefers intermediate over close)')

---

## 5. Training Integration — GRPO + TRL

The `openenv_reward()` function is the integration point with TRL's GRPOTrainer.
It receives LLM completion texts and returns reward scalars.

In [ ]:
from server.email_env.email_negotiation import EmailNegotiationCore

def parse_action_from_text(text: str) -> dict:
    """Parse LLM completion text into an action dict.
    
    In production this uses a more sophisticated parser.
    Here we do simple keyword matching.
    """
    text_lower = text.lower()
    
    # Intent
    intent = 'address_concern'
    if 'offer' in text_lower and 'document' in text_lower:
        intent = 'offer_document'
    elif 'concession' in text_lower or 'reduce' in text_lower:
        intent = 'make_concession'
    elif 'escalate' in text_lower or 'executive' in text_lower:
        intent = 'escalate_to_exec'
    elif 'group' in text_lower or 'all stakeholders' in text_lower:
        intent = 'group_proposal'
    elif 'walk' in text_lower or 'cancel' in text_lower:
        intent = 'walkaway'
    
    # Target
    target = 'Legal'
    for t in ['legal', 'finance', 'cto', 'procurement', 'operations', 'exec']:
        if t in text_lower:
            target = t.title() if t != 'cto' else 'CTO'
            break
    
    # Document
    doc = None
    for d in ['DPA', 'roi_model', 'security_cert', 'implementation_timeline', 'vendor_packet']:
        if d.replace('_', ' ').lower() in text_lower or d.lower() in text_lower:
            doc = d
            break
    
    return {
        'intent': intent,
        'target': target,
        'tone': 'urgent' if 'urgent' in text_lower else 'reassuring' if 'reassur' in text_lower else 'formal',
        'cc': [],
        'include_document': doc,
        'concession_term': 'price' if 'price' in text_lower else None,
        'concession_value': None,
        'urgency': 'high' if 'urgent' in text_lower else 'normal',
    }

In [ ]:
def openenv_reward(completions, **kwargs):
    """
    Called by TRL's GRPOTrainer with LLM completion texts.
    Returns list of reward scalars.
    
    Args:
        completions: list of LLM-generated text strings
    Returns:
        list of float reward values (one per completion)
    """
    rewards = []
    for completion in completions:
        env = EmailNegotiationCore(scenario_type='aligned', use_buyer_llm=False)
        env.reset()
        
        action_dict = parse_action_from_text(str(completion))
        result = env.step(action_dict)
        rewards.append(result['reward'])
    return rewards

In [ ]:
# Test openenv_reward with simulated completions
simulated_completions = [
    'I would like to offer the DPA document to Legal for review.',
    'We need to escalate this to the executive sponsor immediately.',
    'Finance is asking for ROI analysis. I will attach the roi_model.',
    'We can offer a 15% price reduction as a concession.',
    'Addressing the compliance concern raised by Legal regarding liability.',
]

rewards = openenv_reward(simulated_completions)
for comp, r in zip(simulated_completions, rewards):
    print(f'Reward {r:+.3f} | {comp[:60]}')

---

## 6. Training Loop — 50 Steps with Reward Curves

We simulate GRPO-style training by running episodes and collecting rewards.
In production, the GRPOTrainer would call `openenv_reward()` for each batch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def run_training_episode(scenario='aligned', max_steps=10):
    """Run one episode and return reward/progress history."""
    env = EmailNegotiationCore(scenario_type=scenario, use_buyer_llm=False)
    env.reset()
    
    rewards = []
    progress = []
    
    # Action templates — the "policy" we're training
    policy = [
        {'intent': 'offer_document', 'target': 'Legal',    'include_document': 'DPA'},
        {'intent': 'offer_document', 'target': 'Finance',  'include_document': 'roi_model'},
        {'intent': 'make_concession', 'target': 'Legal',    'concession_term': 'price', 'concession_value': 85000},
        {'intent': 'address_concern', 'target': 'CTO',      'include_document': 'security_cert'},
        {'intent': 'address_concern', 'target': 'Finance',  'include_document': None},
        {'intent': 'make_concession', 'target': 'Legal',    'concession_term': 'timeline'},
        {'intent': 'group_proposal',  'target': 'Legal',    'cc': ['Finance', 'CTO']},
        {'intent': 'address_concern', 'target': 'Finance',  'include_document': None},
        {'intent': 'offer_document', 'target': 'Legal',     'include_document': 'implementation_timeline'},
        {'intent': 'escalate_to_exec', 'target': 'ExecSponsor'},
    ]
    
    for i in range(max_steps):
        action_dict = {**policy[i % len(policy)], 'tone': 'formal', 'urgency': 'normal'}
        result = env.step(action_dict)
        rewards.append(result['reward'])
        progress.append(result['progress_score'])
        if result['done']:
            break
    
    return rewards, progress

# Run 50 training episodes
all_episode_rewards = []
all_episode_final_progress = []
all_episode_outcomes = []

print('Running 50 training episodes...')
for episode in range(50):
    scenario = np.random.choice(['aligned', 'aligned', 'conflicted', 'hostile_acquisition'], 
                              p=[0.5, 0.3, 0.15, 0.05])
    rewards, progress = run_training_episode(scenario=scenario)
    all_episode_rewards.append(sum(rewards))
    all_episode_final_progress.append(progress[-1] if progress else 0)
    state = {'terminal_outcome': None}
    all_episode_outcomes.append('in_progress')

print(f'Done. Mean episode reward: {np.mean(all_episode_rewards):.3f}')
print(f'Mean final progress:       {np.mean(all_episode_final_progress):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Cumulative reward per episode
ax = axes[0]
ax.plot(all_episode_rewards, color='steelblue', alpha=0.7, lw=1.5)
# Smoothed
window = 10
smoothed = np.convolve(all_episode_rewards, np.ones(window)/window, mode='valid')
ax.plot(range(window-1, 50), smoothed, color='navy', lw=2.5, label=f'{window}-ep moving avg')
ax.axhline(0, color='gray', lw=1)
ax.set_title('Cumulative Reward per Episode')
ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward')
ax.legend(); ax.grid(True, alpha=0.3)

# 2. Final progress score per episode
ax = axes[1]
ax.scatter(range(50), all_episode_final_progress, alpha=0.5, color='green', s=20)
smoothed_p = np.convolve(all_episode_final_progress, np.ones(window)/window, mode='valid')
ax.plot(range(window-1, 50), smoothed_p, color='darkgreen', lw=2.5, label=f'{window}-ep moving avg')
ax.set_title('Final Progress Score per Episode')
ax.set_xlabel('Episode'); ax.set_ylabel('Progress Score (S×D×G×R)')
ax.set_ylim(0, 1.1); ax.legend(); ax.grid(True, alpha=0.3)

# 3. Distribution of episode total rewards
ax = axes[2]
ax.hist(all_episode_rewards, bins=15, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(np.mean(all_episode_rewards), color='red', lw=2, label=f'mean={np.mean(all_episode_rewards):.2f}')
ax.set_title('Distribution of Episode Rewards')
ax.set_xlabel('Total Reward'); ax.set_ylabel('Count')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Training Summary ===')
print(f'Total episodes:  50')
print(f'Mean reward:     {np.mean(all_episode_rewards):.3f} ± {np.std(all_episode_rewards):.3f}')
print(f'Min/Max reward:  {min(all_episode_rewards):.3f} / {max(all_episode_rewards):.3f}')
print(f'Mean progress:   {np.mean(all_episode_final_progress):.3f}')
print('Saved: training_curves.png')

---

## 7. Colab Deployment — Connect to HF Spaces

After `openenv push`, replace the local core with the remote client:

```python
# Replace:
# from server.email_env.email_negotiation import EmailNegotiationCore

# With:
from client import EmailNegotiationEnv  # HTTPEnvClient subclass

SPACE_URL = "https://YOUR_USERNAME-email-negotiation.hf.space"

def openenv_reward_remote(completions, **kwargs):
    rewards = []
    with EmailNegotiationEnv(base_url=SPACE_URL).sync() as env:
        env.reset()
        for completion in completions:
            action = parse_action_from_text(str(completion))
            result = env.step(action)
            rewards.append(result.reward)
    return rewards
```

Then pass `reward_funcs=[openenv_reward_remote]` to GRPOTrainer.

In [ ]:
# Demonstrate the client interface (connects to local server)
from client import EmailNegotiationEnv

# NOTE: In Colab without Docker running, use EmailNegotiationCore directly.
# The client is for when the server is running (openenv push → HF Spaces).

print('EmailNegotiationEnv client:')
print('  action_class:', EmailNegotiationEnv.action_class)
print('  observation_class:', EmailNegotiationEnv.observation_class)
print('  state_class:', EmailNegotiationEnv.state_class)
print()
print('In Colab, use EmailNegotiationCore directly for local training.')
print('Use EmailNegotiationEnv only when connected to a running HF Space.')

---

## 8. GRPO Training Code (for production use)

This is the actual TRL integration pattern:

```python
# from trl import GRPOTrainer, GRPOConfig
# from transformers import AutoModelForCausalLM, AutoTokenizer
#
# model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
# tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
#
# training_config = GRPOConfig(
#     output_dir="./output",
#     num_train_epochs=3,
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=2,
#     learning_rate=1e-5,
#     bf16=True,
# )
#
# trainer = GRPOTrainer(
#     model=model,
#     args=training_config,
#     train_dataset=email_negotiation_dataset,  # Your dataset
#     reward_funcs=[openenv_reward],           # <-- Our function
# )
#
# trainer.train()
```

---

## 9. Anti-Gaming Analysis

Three-layer defense ensures reward integrity.

In [ ]:
from server.email_env.anti_gaming import AntiGamingValidator

validator = AntiGamingValidator()

print('=== Anti-Gaming Layer 1: Policy Constraints ===')
valid, reason = validator.validate_action({'intent': 'walkaway', 'target': 'Legal', 'tone': 'formal', 'cc': [], 'include_document': None})
print(f'Valid walkaway: {valid} (no block — legitimate action)')

# Try to violate CC limit
valid, reason = validator.validate_action({
    'intent': 'address_concern', 'target': 'Legal', 'tone': 'formal',
    'cc': ['Finance', 'CTO', 'Procurement'],  # 3 recipients — too many
    'include_document': None
})
print(f'CC=3 rejected: {valid}, reason: {reason}')

# Try identical consecutive action
validator.validate_action({'intent': 'offer_document', 'target': 'Legal', 'tone': 'formal', 'cc': [], 'include_document': 'DPA'})
valid, reason = validator.validate_action({'intent': 'offer_document', 'target': 'Legal', 'tone': 'formal', 'cc': [], 'include_document': 'DPA'})
print(f'Identical consecutive rejected: {valid}, reason: {reason}')

print('\n=== Anti-Gaming Layer 2: Reward Shaping ===')
print('Dense rewards clipped to [-0.3, +0.3]')
print('Milestone rewards clipped to [-0.5, +0.5]')
print('Total dense ≤ 0.5 × terminal reward')

print('\n=== Anti-Gaming Layer 3: Diminishing Returns ===')
r1 = validator.apply_diminishing_returns(0.15, 'sentiment_positive')
r2 = validator.apply_diminishing_returns(0.15, 'sentiment_positive')
r3 = validator.apply_diminishing_returns(0.15, 'sentiment_positive')
print(f'First positive sentiment:  {r1:.3f}')
print(f'Second positive sentiment:  {r2:.3f} (×0.9)')
print(f'Third positive sentiment:   {r3:.3f} (×0.9²)')

---

## 10. End-to-End Scenario: Conflicted Deal

Run one full episode through a conflicted scenario and analyze the full trace.

In [ ]:
print('=== Conflicted Scenario — Full Episode Trace ===\n')

env = EmailNegotiationCore(scenario_type='conflicted', use_buyer_llm=False)
obs = env.reset()

print(f'Scenario: conflicted')
print(f'Initial concerns: {obs["unresolved_concerns"]}')
print(f'Initial progress: {obs["progress_score"]:.3f}')
print()

policy = [
    {'intent': 'offer_document', 'target': 'Legal',    'include_document': 'DPA',         'tone': 'formal'},
    {'intent': 'offer_document', 'target': 'Finance',  'include_document': 'roi_model',     'tone': 'reassuring'},
    {'intent': 'make_concession', 'target': 'Finance',  'concession_term': 'price',        'concession_value': 80000, 'tone': 'formal'},
    {'intent': 'address_concern', 'target': 'CTO',      'include_document': 'security_cert', 'tone': 'formal'},
    {'intent': 'make_concession', 'target': 'Legal',    'concession_term': 'liability_cap', 'concession_value': 500000, 'tone': 'reassuring'},
    {'intent': 'group_proposal',  'target': 'Legal',    'cc': ['Finance', 'CTO'],          'tone': 'formal'},
    {'intent': 'address_concern', 'target': 'Finance',  'include_document': None,          'tone': 'reassuring'},
    {'intent': 'offer_document', 'target': 'CTO',      'include_document': 'implementation_timeline', 'tone': 'formal'},
    {'intent': 'address_concern', 'target': 'Procurement', 'include_document': None,       'tone': 'formal'},
    {'intent': 'escalate_to_exec', 'target': 'ExecSponsor', 'tone': 'urgent'},
]

total_reward = 0
for i, base_action in enumerate(policy):
    action_dict = {
        **base_action, 
        'cc': base_action.get('cc', []),
        'urgency': base_action.get('urgency', 'normal')
    }
    result = env.step(action_dict)
    total_reward += result['reward']
    
    state = env.get_state()
    outcome = state.get('terminal_outcome', 'in_progress')
    
    print(f'Step {i+1}: {action_dict["intent"]:<20} → {action_dict["target"]:<12} '
          f'| reward {result["reward"]:+.2f} '
          f'| progress {result["progress_score"]:.3f} '
          f'| stage {result["deal_stage"]}')
    
    if result['done']:
        print(f'\n  → TERMINAL: {outcome}')
        break

print(f'\nTotal reward: {total_reward:.3f}')
print(f'Final outcome: {env.get_state().get("terminal_outcome", "incomplete")}')

In [ ]:
import json

print('\n=== Final State JSON ===')
state = env.get_state()
print(json.dumps({k: v for k, v in state.items() if k != 'alignment_scores'}, indent=2))

print('\n=== Alignment Scores (S component input) ===')
for sid, score in sorted(state.get('alignment_scores', {}).items(), key=lambda x: -x[1]):
    w = STAKEHOLDER_WEIGHTS.get(sid, 0)
    print(f'  {sid:<15}: {score:.3f} (weight={w:.2f})')